# 00 · Stream & explore an OpenScope NWB session

Entry point for DANDI:000253 (Global/Local Oddball). Streams one session over S3 — no 680 GB download.
Then sketches the deviant-vs-standard contrast and the MNE bridge that will carry over to human EEG.

See `docs/02-mouse-data.md`, `docs/03-analysis-pipelines.md`, `docs/05-cross-species-analysis.md`.

In [ ]:
# pip install dandi pynwb h5py remfile mne
import h5py, remfile
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient

DANDISET, VERSION, INDEX = "000253", "draft", 0
with DandiAPIClient() as client:
    ds = client.get_dandiset(DANDISET, VERSION)
    assets = list(ds.get_assets())
    asset = assets[INDEX]
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
print(len(assets), "assets;", asset.path)

In [ ]:
# Lazy HDF5 over HTTP — only metadata + requested chunks are fetched.
rem = remfile.File(s3_url)
h5 = h5py.File(rem, "r")
nwb = NWBHDF5IO(file=h5, mode="r", load_namespaces=True).read()
print(nwb.session_description)
print("acquisition:", list(nwb.acquisition.keys()))
print("processing:", list(nwb.processing.keys()))
print("intervals:", list(nwb.intervals.keys()) if nwb.intervals else None)
print("units:", None if nwb.units is None else len(nwb.units))

## Inspect the stimulus / trials table
Find the oddball structure (standard vs deviant, local `xxxY` vs global `xxxX`). Column names vary by
session — list them first, then build the epoch masks.

In [ ]:
# Example: pull the first stimulus/trials interval table to a DataFrame (small).
if nwb.intervals:
    name = list(nwb.intervals.keys())[0]
    df = nwb.intervals[name].to_dataframe()
    print(name, df.shape)
    display(df.head())
    print(list(df.columns))

## The cross-species bridge (sketch)
The mouse deviant-minus-standard LFP/PSTH and the band-limited (theta / alpha-beta / gamma) time-frequency
maps are the SAME contrasts you compute on human EEG with MNE. Keep one analysis-spec (bands, windows,
baseline, cluster stats) shared between this notebook and an MNE script — when the human EEG arrives you
change only the reader. Port the Bastos-lab `epych` spectral pipeline (FOOOF + CSD) for the mouse side.

In [ ]:
# Shared analysis spec — used by both the mouse pipeline and the MNE/EEG pipeline.
ANALYSIS_SPEC = {
    "bands": {"theta": (2, 10), "alpha_beta": (10, 30), "gamma": (30, 90)},
    "epoch": {"tmin": -0.2, "tmax": 0.5, "baseline": (-0.2, 0.0)},
    "contrasts": ["deviant_minus_standard", "global_minus_local", "omission"],
    "stats": {"method": "cluster_permutation", "alpha": 0.01},
}
ANALYSIS_SPEC